In [ ]:
import pandas as pd

def create_powerworld_aux_final(bus_file, generator_file, line_file, output_file):
    """
    Generate a PowerWorld AUX file with all required fields.
    """
    
    # Read all CSV files
    bus_df = pd.read_csv(bus_file, delimiter=';')
    gen_df = pd.read_csv(generator_file, delimiter=';')
    line_df = pd.read_csv(line_file, delimiter=';')
    
    # Function to format number with comma as decimal separator
    def format_num(value, decimals=4):
        if pd.isna(value):
            return "0,0"
        # Convert to string and replace dot with comma
        return str(round(float(value), decimals)).replace('.', ',')
    
    with open(output_file, 'w', encoding='utf-8') as f:
        
        # ============================================
        # BUS DATA
        # ============================================
        f.write("Bus([Number, Name, NomkV, AreaNum, ZoneNum, Vpu, Vangle]) {\n")
        
        for _, bus in bus_df.iterrows():
            bus_num = int(bus['bus_i'])
            base_kv = float(bus['baseKV'])
            
            # Find matching generator for voltage setpoint
            gen_match = gen_df[gen_df['bus'] == bus_num]
            if not gen_match.empty:
                pu_volt = float(gen_match.iloc[0]['Vg'])
            else:
                pu_volt = 1.0
            
            f.write(f'{bus_num} "Bus {bus_num}" {format_num(base_kv,1)} 1 1 {format_num(pu_volt,3)} 0,0\n')
        
        f.write("}\n\n")
        
        # ============================================
        # LOAD DATA
        # ============================================
        f.write("Load([BusNum, LoadID, Status, SMW, SMvar]) {\n")
        
        load_id = 1
        for _, bus in bus_df.iterrows():
            bus_num = int(bus['bus_i'])
            pd_val = float(bus['Pd']) if pd.notna(bus['Pd']) and float(bus['Pd']) > 0 else 0
            qd_val = float(bus['Qd']) if pd.notna(bus['Qd']) and float(bus['Qd']) > 0 else 0
            
            if pd_val > 0 or qd_val > 0:
                f.write(f'{bus_num} "{load_id}" 1 {format_num(pd_val,1)} {format_num(qd_val,1)}\n')
                load_id += 1
        
        f.write("}\n\n")
        
        # ============================================
        # GENERATOR DATA - Added AGC and AVR fields
        # ============================================
        f.write("Gen([BusNum, ID, Status, MWSetPoint, MvarSetPoint, MWMin, MWMax, VoltSet, MvarMin, MvarMax, MVABase, AGC, AVR]) {\n")
        
        for _, gen in gen_df.iterrows():
            bus_num = int(gen['bus'])
            pg = float(gen['Pg'])
            qg = float(gen['Qg'])
            vg = float(gen['Vg'])
            pmax = float(gen['Pmax']) if pd.notna(gen['Pmax']) else pg * 1.5
            pmin = float(gen['Pmin']) if pd.notna(gen['Pmin']) else 10.0
            qmax = float(gen['Qmax']) if pd.notna(gen['Qmax']) else 300.0
            qmin = float(gen['Qmin']) if pd.notna(gen['Qmin']) else -300.0
            
            # Add AGC and AVR flags (1 = enabled)
            f.write(f'{bus_num} "1" 1 {format_num(pg,1)} {format_num(qg,1)} {format_num(pmin,1)} {format_num(pmax,1)} {format_num(vg,3)} {format_num(qmin,1)} {format_num(qmax,1)} 100,0 1 1\n')
        
        f.write("}\n\n")
        
        # ============================================
        # BRANCH DATA - Added LimitMVAB and LimitMVAC fields
        # ============================================
        f.write("Branch([BusNumFrom, BusNumTo, Circuit, Status, R, X, B, LimitMVAA, LimitMVAB, LimitMVAC]) {\n")
        
        for _, line in line_df.iterrows():
            fbus = int(line['fbus'])
            tbus = int(line['tbus'])
            r = float(line['r'])
            x = float(line['x'])
            b = float(line['b'])
            rateA = float(line['rateA']) if pd.notna(line['rateA']) else 250.0
            rateB = float(line['rateB']) if pd.notna(line['rateB']) else rateA
            rateC = float(line['rateC']) if pd.notna(line['rateC']) else rateA
            
            f.write(f'{fbus} {tbus} "1" 1 {format_num(r,4)} {format_num(x,4)} {format_num(b,3)} {format_num(rateA,1)} {format_num(rateB,1)} {format_num(rateC,1)}\n')
        
        f.write("}\n")
    
    print(f"Successfully created AUX file with all required fields: {output_file}")
    print("\nChanges made:")
    print("  - Added AGC and AVR fields to generators (set to 1)")
    print("  - Added LimitMVAB and LimitMVAC fields to branches (using same values as LimitMVAA)")

# Run the script
if __name__ == "__main__":
    create_powerworld_aux_final(
        "bus_data.csv",
        "generators_data.csv",
        "line_data.csv",
        "power_system_final.aux"
    )

Successfully created corrected AUX file: power_system_corrected.aux
